# Apply Curation

Applica le sostituzioni individuate da `05_answer_curation.ipynb`.

**Input** (tutti file esistenti, nessuno viene sovrascritto):
- `top100_merged.parquet` — top-100 originali (1.000 query)
- `top100_candidates.parquet` — top-100 delle 5.000 candidate
- `passage_entities.parquet` + `curation_chunks/*.parquet` — entità dei passaggi
- `query_embeddings.npy` — embedding delle 1.000 query originali
- `queries_subset.jsonl` — metadati delle 1.000 query originali
- `curation_results.jsonl` — mapping delle 344 sostitute valide
- `qa_all_entities.jsonl` — pool completo delle query NQ (per metadati sostitute)

**Output** (file nuovi):
- `top100_curated.parquet`
- `passage_entities_curated.parquet`
- `query_embeddings_curated.npy`
- `queries_curated.jsonl`

In [6]:
import json          # json.loads: parsa una stringa JSON → dict/list Python
from pathlib import Path  # Path: gestione cross-platform dei percorsi file

import numpy as np   # np.load/np.save: carica/salva array numerici in formato .npy
import polars as pl  # polars: libreria DataFrame colonnare (alternativa veloce a pandas)

# Directory di input/output — tutti i file letti e scritti stanno qui
OUTPUT_DIR   = Path("data/NQ_answer")    # top-100, entità passaggi, embedding, metadati query
QUESTION_DIR = Path("data/NQ_question")  # pool completo delle domande Natural Questions

## 1. Load Data

In [ ]:
# =============================================================================
# 1a. Top-100 originali
# =============================================================================
# pl.read_parquet: legge un file Parquet (formato colonnare binario) → DataFrame Polars
# Input:  top100_merged.parquet — 1.000 query × 100 passaggi = 100.000 righe
# Colonne principali: query_id (Int32), passage_id, rank, score
# Output: DataFrame `top100` con tutti i risultati di retrieval originali
top100 = pl.read_parquet(OUTPUT_DIR / "top100_merged.parquet")
# .height → numero di righe del DataFrame (equivalente di len(df) in pandas)
# .n_unique() → conta i valori distinti nella colonna (come nunique() in pandas)
print(f"top100_merged:        {top100.height:,} rows, {top100['query_id'].n_unique()} queries")

# =============================================================================
# 1b. Top-100 candidate (pool allargato per la curation)
# =============================================================================
# Input:  top100_candidates.parquet — 5.000 query candidate × 100 passaggi = 500.000 righe
#         Queste sono query NQ aggiuntive da cui pescare le 344 sostitute
# Output: DataFrame `cand_top100` con la stessa struttura di top100
cand_top100 = pl.read_parquet(OUTPUT_DIR / "top100_candidates.parquet")
print(f"top100_candidates:    {cand_top100.height:,} rows, {cand_top100['query_id'].n_unique()} queries")

# =============================================================================
# 1c. Entità dei passaggi (originali + chunk della curation)
# =============================================================================
# Ogni riga = un passaggio con le sue entità Wikidata linkate
# Colonne: id (passage_id), qids (List[str] di Wikidata QID)
psg_ent_orig = pl.read_parquet(OUTPUT_DIR / "passage_entities.parquet")

# pl.read_parquet con una directory: legge TUTTI i .parquet nella cartella
# e li concatena automaticamente in un unico DataFrame
curation_chunks = pl.read_parquet(OUTPUT_DIR / "curation_chunks/")

# pl.concat: concatena verticalmente (impila le righe) due DataFrame
# .unique(subset=["id"]): rimuove righe duplicate basandosi sulla colonna "id",
#   necessario perché alcuni passaggi possono apparire sia nell'originale che nei chunk
# Output: psg_ent_all contiene le entità di TUTTI i passaggi (originali + candidati)
psg_ent_all = pl.concat([psg_ent_orig, curation_chunks]).unique(subset=["id"])
print(f"passage_entities:     {psg_ent_orig.height:,} (original) + "
      f"{curation_chunks.height:,} (curation) = {psg_ent_all.height:,} (unique)")

# =============================================================================
# 1d. Embedding delle query originali
# =============================================================================
# np.load: carica un array NumPy salvato in formato .npy (binario, veloce)
# Input:  query_embeddings.npy — matrice (1000, 768)
#         Ogni riga è il vettore Contriever a 768 dimensioni di una query
#         L'indice della riga corrisponde al query_id (riga 0 = query_id 0)
orig_emb = np.load(str(OUTPUT_DIR / "query_embeddings.npy"))
print(f"query_embeddings:     {orig_emb.shape}")

# =============================================================================
# 1e. Mapping delle sostituzioni (output di 05_answer_curation.ipynb)
# =============================================================================
# Formato JSONL: un oggetto JSON per riga
# Ogni oggetto contiene:
#   - "local_query_id": id della query nel pool candidato (indice in cand_top100)
#   - "original_query_id": id della query nel pool NQ completo (indice in qa_all_entities)
# Output: lista di 344 dict, uno per ogni sostituzione da applicare
with open(OUTPUT_DIR / "curation_results.jsonl", encoding="utf-8") as f:
    subs = [json.loads(line) for line in f]
print(f"curation_results:     {len(subs)} substitutes")

## 2. Identify Queries to Replace

Ricalcoliamo le query problematiche direttamente dai dati: sono quelle che hanno
almeno un passaggio con 0 entità nella top-100 originale.

In [ ]:
# =============================================================================
# Obiettivo: trovare quali query_id hanno almeno un passaggio con 0 entità
# Queste sono le query "problematiche" che verranno sostituite
# =============================================================================

# .select(): crea un nuovo DataFrame con solo le colonne specificate
# pl.col("qids").list.len(): per ogni riga, conta quanti elementi ha la lista "qids"
#   qids è una colonna di tipo List[str] contenente i Wikidata QID del passaggio
#   .list è l'accessor per operazioni su colonne di tipo List
#   .len() conta gli elementi della lista → risultato: intero (0 se lista vuota)
# .alias("n_qids"): rinomina la colonna risultante in "n_qids"
# Output: DataFrame con colonne [id, n_qids] — un conteggio entità per passaggio
ent_counts = psg_ent_orig.select("id", pl.col("qids").list.len().alias("n_qids"))

# .join(): SQL-like join tra due DataFrame
#   left_on="passage_id": colonna del DataFrame sinistro (top100) da matchare
#   right_on="id": colonna del DataFrame destro (ent_counts) da matchare
#   how="left": LEFT JOIN — mantiene tutte le righe di top100, aggiunge n_qids
#     se un passage_id non ha match in ent_counts, n_qids sarà null
# Output: top100 arricchito con la colonna n_qids (numero entità per passaggio)
top100_ent = top100.join(ent_counts, left_on="passage_id", right_on="id", how="left")

# .filter(): seleziona solo le righe che soddisfano la condizione
#   pl.col("n_qids") == 0: passaggi che non hanno nessuna entità linkata
# ["query_id"]: estrae solo la colonna query_id come Series
# .unique(): rimuove duplicati (una query può avere più passaggi con 0 entità)
# .to_list(): converte la Series Polars → lista Python
# set(): converte in set per lookup O(1) — usato dopo per filtrare velocemente
queries_to_replace = set(
    top100_ent.filter(pl.col("n_qids") == 0)["query_id"].unique().to_list()
)

n_total = top100["query_id"].n_unique()
print(f"Queries to replace: {len(queries_to_replace)} / {n_total}")

# Sanity check: il numero di query problematiche deve coincidere con
# il numero di sostitute trovate in 05_answer_curation.ipynb
assert len(queries_to_replace) == len(subs), (
    f"Mismatch: {len(queries_to_replace)} bad queries vs {len(subs)} substitutes"
)

## 3. Apply Substitution

Ogni sostituta **eredita il query_id** della query che rimpiazza, così il range 0–999 resta intatto.

In [9]:
# =============================================================================
# Obiettivo: sostituire le 344 query problematiche con query candidate "buone",
# mantenendo il range query_id 0–999 intatto (ogni sostituta eredita l'id della
# query che rimpiazza)
# =============================================================================

# sorted(): ordina i query_id problematici in ordine crescente
# Serve per l'accoppiamento 1:1 con la lista `subs` (anch'essa ordinata)
bad_qids = sorted(queries_to_replace)

# Costruisce il dizionario di rimappatura:
#   chiave = local_query_id della candidata (il suo id dentro cand_top100)
#   valore = query_id che deve ereditare (lo slot 0–999 della query rimossa)
# zip(bad_qids, subs): accoppia il k-esimo bad_qid con il k-esimo sostituto
# Esempio: {4523: 7, 1890: 23, ...} → la candidata 4523 diventa query_id 7
sub_lid_to_qid = {
    s["local_query_id"]: bqid
    for bqid, s in zip(bad_qids, subs)
}

# =============================================================================
# 3a. Righe originali "buone" (query che NON vengono sostituite)
# =============================================================================
# .filter(): mantiene solo le righe dove query_id NON è tra i bad_qids
# ~: operatore NOT in Polars (nega la condizione)
# .is_in(bad_qids): restituisce True se il valore è presente nella lista
# Output: DataFrame con le righe delle 656 query buone (65.600 righe)
top100_good = top100.filter(~pl.col("query_id").is_in(bad_qids))

# =============================================================================
# 3b. Righe sostitutive: prendi le candidate e rimappa il loro query_id
# =============================================================================
sub_local_ids = list(sub_lid_to_qid.keys())

top100_subs = (
    cand_top100
    # Filtra solo le 344 query candidate che sono state selezionate come sostitute
    .filter(pl.col("query_id").is_in(sub_local_ids))
    .with_columns(
        # .replace(sub_lid_to_qid): rimappa i valori della colonna usando il dizionario
        #   Per ogni valore nella colonna, se è una chiave del dict, lo sostituisce col valore
        #   Esempio: query_id 4523 → 7 (eredita lo slot della query rimossa)
        # .cast(pl.Int32): converte il tipo da Int64 a Int32
        #   Necessario perché .replace() restituisce Int64 (i valori Python int sono 64-bit)
        #   ma top100_good ha query_id come Int32 — pl.concat richiede tipi identici
        pl.col("query_id").replace(sub_lid_to_qid).cast(pl.Int32).alias("query_id")
    )
)

# Allinea tipi colonna (es. Float64→Float32) per evitare SchemaError nel concat
top100_subs = top100_subs.cast({col: dtype for col, dtype in top100_good.schema.items()})


# =============================================================================
# 3c. Concatenazione: unisce righe buone + righe sostitutive
# =============================================================================
# pl.concat([df1, df2]): impila verticalmente i due DataFrame (come UNION ALL in SQL)
#   Richiede che entrambi abbiano le stesse colonne con gli stessi tipi
# .sort("query_id", "rank"): ordina prima per query_id, poi per rank dentro ogni query
# Output: DataFrame con 100.000 righe (1.000 query × 100 passaggi)
top100_curated = pl.concat([top100_good, top100_subs]).sort("query_id", "rank")

print(f"top100_curated: {top100_curated.height:,} rows, "
      f"{top100_curated['query_id'].n_unique()} queries")
print(f"  good original: {top100_good.height:,} rows ({top100_good['query_id'].n_unique()} queries)")
print(f"  substituted:   {top100_subs.height:,} rows ({top100_subs['query_id'].n_unique()} queries)")

top100_curated: 100,000 rows, 1000 queries
  good original: 65,600 rows (656 queries)
  substituted:   34,400 rows (344 queries)


## 4. Curated Passage Entities

Filtriamo solo i passaggi effettivamente referenziati dal top-100 curato.

In [10]:
# =============================================================================
# Obiettivo: creare un file di entità "curato" che contiene SOLO i passaggi
# effettivamente presenti nel top-100 curato (non tutto il pool)
# =============================================================================

# ["passage_id"]: estrae la colonna passage_id come Series Polars
# .unique(): rimuove duplicati — un passage_id può comparire in più query
# Output: Series con tutti i passage_id distinti nel dataset curato
curated_pids = top100_curated["passage_id"].unique()

# .filter(pl.col("id").is_in(curated_pids)): mantiene solo le righe il cui
#   id (passage_id nella tabella entità) è presente nel set dei passaggi curati
# Input:  psg_ent_all — tutte le entità (originali + candidati), ~467k righe
# Output: psg_ent_curated — solo le entità dei passaggi usati, molte meno righe
psg_ent_curated = psg_ent_all.filter(pl.col("id").is_in(curated_pids))

# Verifica completezza: OGNI passaggio nel top-100 curato deve avere dati entità
# Se mancano, il knowledge graph a valle sarà incompleto
# .filter(~...is_in(...)): trova i passage_id del curato che NON hanno un match
#   nelle entità — idealmente questa lista è vuota
missing = curated_pids.filter(~curated_pids.is_in(psg_ent_curated["id"]))
print(f"Passages in curated top-100:   {curated_pids.len():,}")
print(f"Passages with entity data:     {psg_ent_curated.height:,}")
print(f"Missing entity data:           {missing.len()}")

if missing.len() > 0:
    print("\nWARNING: some passages lack entity data!")

Passages in curated top-100:   90,667
Passages with entity data:     90,667
Missing entity data:           0


C:\Users\Utente\AppData\Local\Temp\ipykernel_10324\280628530.py:15: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  psg_ent_curated = psg_ent_all.filter(pl.col("id").is_in(curated_pids))
C:\Users\Utente\AppData\Local\Temp\ipykernel_10324\280628530.py:21: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  missing = curated_pids.filter(~curated_pids.is_in(psg_ent_curated["id"]))


## 5. Curated Query Embeddings

Per le 656 query buone usiamo gli embedding originali.  
Per le 344 sostitute serve ricodificare con Contriever.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModel  # HuggingFace Transformers

# =============================================================================
# 5a. Carica i testi delle domande sostitutive
# =============================================================================
# qa_all_entities.jsonl: pool completo di ~87k domande Natural Questions
# Ogni riga JSON ha: question, answers, question_qids, answer_variant_qids
# Serve per recuperare il testo della domanda delle 344 sostitute
with open(QUESTION_DIR / "qa_all_entities.jsonl", encoding="utf-8") as f:
    all_queries = [json.loads(line) for line in f]

# Per ogni sostituzione, s["original_query_id"] è l'indice nel pool completo
# Recuperiamo il testo "question" di ciascuna sostituta
sub_questions = [all_queries[s["original_query_id"]]["question"] for s in subs]
print(f"Substitute questions to encode: {len(sub_questions)}")

# =============================================================================
# 5b. Mean pooling — stessa funzione usata in 04_answer_preparation e 03_embedding.ipynb
# =============================================================================
# Contriever non ha un token [CLS] dedicato, usa mean pooling su tutti i token
def mean_pool(outputs, attention_mask):
    """
    Calcola la media pesata degli hidden states, mascherando i token di padding.

    Input:
      outputs: output del modello transformer
        outputs.last_hidden_state: (batch_size, seq_len, hidden_dim=768)
      attention_mask: (batch_size, seq_len) — 1 per token reali, 0 per padding

    Operazioni:
      1. unsqueeze(-1): (B, L) → (B, L, 1) — aggiunge dimensione per broadcasting
      2. hidden * mask: azzera gli hidden states dei token di padding
      3. .sum(dim=1): somma lungo l'asse dei token → (B, 768)
      4. / mask.sum(dim=1): divide per il numero di token reali (non padding)

    Output: (batch_size, 768) — un vettore embedding per frase
    """
    hidden = outputs.last_hidden_state  # (B, L, 768)
    mask = attention_mask.unsqueeze(-1).float()  # (B, L, 1)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1)

# =============================================================================
# 5c. Encoding con Contriever (facebook/contriever)
# =============================================================================
# Scelta device: usa GPU se disponibile, altrimenti CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# AutoTokenizer.from_pretrained: carica il tokenizer del modello
#   Converte testo → token_ids (input numerici per il modello)
tokenizer = AutoTokenizer.from_pretrained("facebook/contriever")

# AutoModel.from_pretrained: carica i pesi del modello Contriever
#   .to(device): sposta i pesi su GPU/CPU
#   .eval(): mette il modello in modalità inferenza (disabilita dropout, batchnorm training)
model = AutoModel.from_pretrained("facebook/contriever").to(device).eval()

BATCH_SIZE = 128  # processa 128 domande alla volta (bilancio memoria/velocità)
sub_embs = []
t0 = time.time()

# torch.no_grad(): disabilita il calcolo dei gradienti
#   Risparmia memoria GPU (~50%) e velocizza l'inferenza — non serve backprop
with torch.no_grad():
    for i in range(0, len(sub_questions), BATCH_SIZE):
        batch = sub_questions[i : i + BATCH_SIZE]

        # tokenizer(): converte lista di stringhe → dizionario con:
        #   input_ids: (B, L) — id dei token
        #   attention_mask: (B, L) — 1=token reale, 0=padding
        #   padding=True: allinea tutte le frasi alla lunghezza massima del batch
        #   truncation=True: tronca a max_length se la frase è troppo lunga
        #   return_tensors="pt": restituisce tensori PyTorch (non liste)
        #   .to(device): sposta i tensori su GPU/CPU
        tok = tokenizer(batch, padding=True, truncation=True,
                        max_length=512, return_tensors="pt").to(device)

        # model(**tok): forward pass — produce gli hidden states
        out = model(**tok)

        # mean_pool: media pesata degli hidden states → embedding (B, 768)
        emb = mean_pool(out, tok["attention_mask"])

        # .cpu().numpy(): sposta il tensore dalla GPU alla CPU e converte in array NumPy
        sub_embs.append(emb.cpu().numpy())

# np.concatenate: unisce la lista di array (B, 768) in un unico array (344, 768)
#   axis=0: concatena lungo la prima dimensione (le righe)
sub_embs = np.concatenate(sub_embs, axis=0)  # (344, 768)
print(f"Encoded {sub_embs.shape[0]} queries -> {sub_embs.shape} in {time.time()-t0:.1f}s")

# Libera la GPU: elimina il modello dalla memoria e svuota la cache CUDA
del model
torch.cuda.empty_cache()

# =============================================================================
# 5d. Assembla la matrice di embedding curata
# =============================================================================
# Strategia: parti dalla matrice originale (1000, 768), sovrascrivi le 344 righe
# corrispondenti alle query sostituite con i nuovi embedding appena calcolati
#
# orig_emb[query_id] = embedding della query originale
# Dopo: curated_emb[query_id] = embedding della sostituta (per i 344 slot bad)

curated_emb = orig_emb.copy()  # .copy(): copia profonda per non modificare l'originale

for s, new_qid in zip(subs, bad_qids):
    # Trova l'indice del sostituto nell'array sub_embs
    # s["local_query_id"] identifica la query candidata, cerchiamo la sua posizione
    # nella lista subs per sapere quale riga di sub_embs corrisponde
    idx_in_sub = [x["local_query_id"] for x in subs].index(s["local_query_id"])
    # Sovrascrive lo slot new_qid (es. 7) con l'embedding della sostituta
    curated_emb[new_qid] = sub_embs[idx_in_sub]

print(f"curated_emb shape: {curated_emb.shape}")  # deve essere (1000, 768)

## 6. Curated Query Metadata

Ricostruiamo il jsonl con i metadati completi.  
Le query sostituite vengono marcate con `"curated": true`.

In [12]:
# =============================================================================
# Obiettivo: aggiornare i metadati delle query (testo, risposte, entità)
# Le 344 query sostituite ricevono i metadati dalla query NQ di provenienza
# e vengono marcate con "curated": True per tracciabilità
# =============================================================================

# Carica i metadati originali delle 1.000 query
# Ogni riga JSON contiene: question, answers, question_qids, answer_variant_qids
with open(OUTPUT_DIR / "queries_subset.jsonl", encoding="utf-8") as f:
    orig_queries_list = [json.loads(line) for line in f]

# list(): shallow copy — crea una nuova lista ma gli elementi (dict) sono condivisi
# Necessario per non modificare orig_queries_list quando sovrascriviamo gli slot
curated_queries = list(orig_queries_list)

# Per ogni sostituzione, sovrascrive lo slot nella lista curata:
#   - new_qid: l'indice 0–999 della query rimossa (posizione nella lista)
#   - s["original_query_id"]: indice nel pool NQ completo (all_queries)
# I metadati vengono copiati dalla query NQ originale + campo "curated" aggiunto
for s, new_qid in zip(subs, bad_qids):
    src = all_queries[s["original_query_id"]]
    curated_queries[new_qid] = {
        "question":            src["question"],            # testo della domanda
        "answers":             src["answers"],              # lista risposte valide
        "question_qids":       src["question_qids"],        # entità Wikidata nella domanda
        "answer_variant_qids": src["answer_variant_qids"],  # entità Wikidata nelle risposte
        "original_query_id":   s["original_query_id"],      # tracciabilità: da dove viene
        "curated":             True,                        # flag: questa query è stata sostituita
    }

# Verifica: conta quante query hanno il flag "curated"
# .get("curated"): restituisce None (falsy) se la chiave non esiste → non contata
n_curated = sum(1 for q in curated_queries if q.get("curated"))
print(f"queries_curated: {len(curated_queries)} total, {n_curated} substituted")

queries_curated: 1000 total, 344 substituted


## 7. Save

In [13]:
# =============================================================================
# Salvataggio dei 4 artefatti curati
# Nessun file originale viene sovrascritto — i nomi hanno suffisso "_curated"
# =============================================================================

# .write_parquet(): serializza il DataFrame Polars in formato Apache Parquet
#   Parquet è colonnare e compresso — molto più compatto e veloce di CSV
top100_curated.write_parquet(OUTPUT_DIR / "top100_curated.parquet")
psg_ent_curated.write_parquet(OUTPUT_DIR / "passage_entities_curated.parquet")

# np.save(): salva un array NumPy in formato binario .npy
#   str() necessario perché np.save non accetta Path su alcune versioni
np.save(str(OUTPUT_DIR / "query_embeddings_curated.npy"), curated_emb)

# Salvataggio JSONL: un oggetto JSON per riga, leggibile e appendibile
# json.dumps(ensure_ascii=False): mantiene caratteri unicode (es. accenti)
#   senza convertirli in sequenze \uXXXX
with open(OUTPUT_DIR / "queries_curated.jsonl", "w", encoding="utf-8") as f:
    for q in curated_queries:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print("Saved:")
print(f"  {OUTPUT_DIR / 'top100_curated.parquet'}")
print(f"  {OUTPUT_DIR / 'passage_entities_curated.parquet'}")
print(f"  {OUTPUT_DIR / 'query_embeddings_curated.npy'}")
print(f"  {OUTPUT_DIR / 'queries_curated.jsonl'}")

Saved:
  data\NQ_answer\top100_curated.parquet
  data\NQ_answer\passage_entities_curated.parquet
  data\NQ_answer\query_embeddings_curated.npy
  data\NQ_answer\queries_curated.jsonl


## 8. Sanity Check

Verifica che nel dataset curato non ci siano passaggi con 0 entità.

In [14]:
# =============================================================================
# Sanity check finale: verifica che la curation abbia eliminato TUTTI i passaggi
# con 0 entità dal dataset. Se ne rimangono, il processo ha un bug.
# =============================================================================

# Conta le entità per ogni passaggio nel dataset curato
# pl.col("qids").list.len(): numero di Wikidata QID per passaggio
ent_check = psg_ent_curated.select("id", pl.col("qids").list.len().alias("n_qids"))

# LEFT JOIN: associa a ogni riga del top-100 curato il conteggio entità
#   how="left": mantiene tutte le righe di top100_curated
#   Se un passage_id non ha match → n_qids sarà null (passaggio senza dati entità)
curated_with_ent = top100_curated.join(
    ent_check, left_on="passage_id", right_on="id", how="left"
)

# Conta le righe problematiche: passaggi con 0 entità O senza dati entità (null)
# pl.col("n_qids").is_null(): il passaggio non aveva riga in passage_entities
# pl.col("n_qids") == 0: il passaggio ha una riga ma la lista qids è vuota
# |: OR logico in Polars (operatore bitwise usato come booleano sulle espressioni)
zero_remaining = curated_with_ent.filter(
    (pl.col("n_qids") == 0) | pl.col("n_qids").is_null()
).height  # .height: numero di righe che matchano il filtro

print(f"{'='*50}")
print(f"SANITY CHECK")
print(f"{'='*50}")
print(f"Total rows:              {curated_with_ent.height:,}")
print(f"Total queries:           {curated_with_ent['query_id'].n_unique()}")
print(f"0-entity rows remaining: {zero_remaining}")
print()
if zero_remaining == 0:
    print("All passages have at least 1 entity.")
else:
    print(f"WARNING: {zero_remaining} rows still have 0 entities!")

SANITY CHECK
Total rows:              100,000
Total queries:           1000
0-entity rows remaining: 0

All passages have at least 1 entity.
